# TMLR runner — one shard per Kaggle session

Runs every run of **one shard** from `configs/tmlr/shards_w*.json`. Two lanes, one per GPU on **GPU T4 x2**. Each run trains, then is evaluated on the frozen 600 test episodes, or on VAL only for E5-val. Results are written to `/kaggle/working/tmlr_out/<SHARD_ID>/`, zipped, and uploaded to Hugging Face.

**Edit only the control cell.** Then: *Save Version → Save & Run All (Commit)*, close the tab, and check *Versions → Logs* later. Full checklist: `docs/KAGGLE_RUNBOOK.md`.

- **Data:** every dataset this shard needs is prepared once, before the lanes start. A source missing from the attached Kaggle dataset is downloaded at that point (MiniImageNet + TinyImageNet: about 8 minutes).
- **Resuming a shard:** attach this notebook's previous output (*Add Input → Your Work*). Finished runs are restored and skipped.
- **On failure:** the version ends in *Error* only after everything finished has been packed and uploaded.

In [ ]:
# ---- CONTROL CELL: the only cell you edit ---------------------------------
SHARD_FILE   = 'configs/tmlr/shards_w0.json'   # shards_w0 / shards_w1 / shards_w2 / shards_r1 ...
SHARD_ID     = 'W0-01'
GIT_SHA      = 'PUT-THE-40-CHAR-COMMIT-SHA-HERE'   # the pushed commit to run; never edit code in Kaggle
LANES        = 2          # 2 on "GPU T4 x2"; 1 on a single T4
MAX_MINUTES  = 600        # stop STARTING new runs after this (Kaggle kills at 12 h)
NUM_EPISODES = 600        # frozen test episodes; never lower for a real run
PERSIST_LOGITS = True     # per-episode logits (~12 MB/run): makes future re-analysis free
UPLOAD_TO_HF = True       # needs Add-ons -> Secrets -> HF_TOKEN
HF_REPO      = 'YOUR-HF-USER/bpeft-tmlr-runs'
REPO_URL     = 'https://github.com/notAvailable73/thesis.git'

In [ ]:
# ---- 1. environment ---------------------------------------------------------
import os, sys, subprocess, json, time, glob, shutil, hashlib, zipfile
import torch, torchvision
print('python', sys.version.split()[0], '| torch', torch.__version__, '| torchvision', torchvision.__version__,
      '| cuda', torch.version.cuda)
n_gpu = torch.cuda.device_count()
names = [torch.cuda.get_device_name(i) for i in range(n_gpu)]
print('GPUs:', names)
assert n_gpu >= LANES, f'need {LANES} GPU(s), found {n_gpu}: Settings -> Accelerator -> GPU T4 x2'
assert all('T4' in n for n in names), f'T4 only (results must be reproducible across sessions): {names}'

In [ ]:
# ---- 2. code at the pinned commit -------------------------------------------
REPO_DIR = '/kaggle/working/thesis' if os.path.isdir('/kaggle/working') else os.path.abspath('thesis')
if not os.path.isdir(os.path.join(REPO_DIR, '.git')):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
subprocess.run(['git', '-C', REPO_DIR, 'fetch', '--all', '--quiet'], check=True)
subprocess.run(['git', '-C', REPO_DIR, 'checkout', '--quiet', GIT_SHA], check=True)
head = subprocess.run(['git', '-C', REPO_DIR, 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()
assert head == GIT_SHA, f'checked out {head}, expected {GIT_SHA}'
os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
for p in (os.getcwd(), os.path.join(os.getcwd(), 'scripts')):
    if p not in sys.path:
        sys.path.insert(0, p)
from pathlib import Path
REPO = Path(os.getcwd())
OUT = Path('/kaggle/working/tmlr_out') / SHARD_ID if os.path.isdir('/kaggle/working') else REPO / 'tmlr_out' / SHARD_ID
OUT.mkdir(parents=True, exist_ok=True)
print('repo at', head)

In [ ]:
# ---- 3. everything else lives in the repo at GIT_SHA ------------------------
# data staging -> preflight -> one-time data preparation -> restore -> 2 GPU lanes
# -> pack + upload -> verdict. See scripts/tmlr_kaggle_session.py.
exec(open('scripts/tmlr_kaggle_session.py').read())
